# Optimizer capabilities

Synthetic 6-site network exercising every constraint type the optimizer supports. Each cell adds one constraint to the same baseline problem so the effect on the solution is visible.

In [ ]:
!pip install --quiet 'git+https://github.com/ttitamu/mcboms-optimization.git#egg=mcboms[pulp]'

In [ ]:
import pandas as pd
from mcboms.core.optimizer import Optimizer

# 6 sites across 3 regions and 2 facility types.
sites = pd.DataFrame({
    "site_id":       [1, 2, 3, 4, 5, 6],
    "facility_type": ["rural_2lane", "rural_2lane",
                      "urban_arterial", "urban_arterial",
                      "rural_2lane", "rural_2lane"],
    "region":        ["North", "North", "Central", "Central", "South", "South"],
})

# Three alternatives per site: do-nothing (0), minor treatment (1), major treatment (2).
alternatives = pd.DataFrame({
    "site_id":       [1,1,1,  2,2,2,  3,3,3,  4,4,4,  5,5,5,  6,6,6],
    "alt_id":        [0,1,2,  0,1,2,  0,1,2,  0,1,2,  0,1,2,  0,1,2],
    "description":   ["nothing","minor","major"] * 6,
    "total_cost":    [0, 150_000, 380_000,  0, 120_000, 320_000,
                      0, 200_000, 500_000,  0, 180_000, 450_000,
                      0, 100_000, 280_000,  0, 130_000, 360_000],
    "total_benefit": [0, 280_000, 720_000,  0, 240_000, 600_000,
                      0, 380_000, 920_000,  0, 340_000, 850_000,
                      0, 200_000, 540_000,  0, 260_000, 700_000],
})

print(f"{len(sites)} sites, {len(alternatives)} alternatives.")
sites

In [ ]:
# Baseline: budget cap and mutual exclusivity only. No optional constraints.
opt = Optimizer(sites=sites, alternatives=alternatives, budget=1_300_000)
print(opt.solve().summary())

In [ ]:
# Dependency: site 5's 'major' requires site 6 only get 'minor' (not 'major').
# Models cases where committing major work at one site forces scaled-back work at adjacent sites.
opt = Optimizer(
    sites=sites, alternatives=alternatives, budget=1_300_000,
    dependencies=[(5, 2, 6, 1)],
)
print(opt.solve().summary())

In [ ]:
# Conflict: sites 5 and 6 cannot both have 'major' at the same time.
# Models cases where adjacent projects can't both be under construction simultaneously.
opt = Optimizer(
    sites=sites, alternatives=alternatives, budget=1_300_000,
    conflicts=[(5, 2, 6, 2)],
)
print(opt.solve().summary())

In [ ]:
# Minimum BCR: any selected alternative must have benefit/cost >= 2.0.
opt = Optimizer(
    sites=sites, alternatives=alternatives, budget=1_300_000,
    min_bcr=2.0,
)
print(opt.solve().summary())

**Network-level constraints.** Aggregate restrictions on top of the project-level structure — funding streams, geographic balance.

In [ ]:
# Facility-type sub-budgets: rural sites <= $700K, urban <= $600K, total <= $1.3M.
opt = Optimizer(
    sites=sites, alternatives=alternatives, budget=1_300_000,
    facility_budgets={"rural_2lane": 700_000, "urban_arterial": 600_000},
)
print(opt.solve().summary())

In [ ]:
# Regional cap: region Central total spending <= $300K.
opt = Optimizer(
    sites=sites, alternatives=alternatives, budget=1_300_000,
    regional_caps={"Central": 300_000},
)
print(opt.solve().summary())

In [ ]:
# Regional floor: region North receives >= 30% of total budget ($390K).
opt = Optimizer(
    sites=sites, alternatives=alternatives, budget=1_300_000,
    regional_floors={"North": 0.30},
)
print(opt.solve().summary())

In [ ]:
# Both solver backends produce the same answer.
kwargs = dict(
    sites=sites, alternatives=alternatives, budget=1_300_000,
    dependencies=[(5, 2, 6, 1)],
    facility_budgets={"rural_2lane": 700_000, "urban_arterial": 600_000},
    regional_caps={"Central": 300_000},
)

result_pulp = Optimizer(**kwargs).solve(solver="pulp")
print(f"PuLP   net=${result_pulp.net_benefit:,.0f}  solver={result_pulp.solver_used}")

# Gurobi may not be installed in Colab; the auto-detector falls back to PuLP.
try:
    result_gurobi = Optimizer(**kwargs).solve(solver="gurobi")
    print(f"Gurobi net=${result_gurobi.net_benefit:,.0f}  solver={result_gurobi.solver_used}")
    assert abs(result_gurobi.objective_value - result_pulp.objective_value) < 1e-3
    print("Both backends agree.")
except RuntimeError as e:
    print(f"Gurobi not available; auto-detect falls back to PuLP+CBC ({e}).")

In [ ]:
# OptimizationResult.to_markdown() emits report-ready markdown.
print(result_pulp.to_markdown())